In [1]:
import pandas as pd
from tqdm import tqdm
import json
import os

path = 'data/'
version ='X4090D-v025'
os.makedirs(path+'models',exist_ok=True)
os.makedirs(path+f'feature',exist_ok=True)
os.makedirs(path+f'feature_importance',exist_ok=True)
os.makedirs(path+'submissions',exist_ok=True)
os.makedirs(path+'logs',exist_ok=True)

fold='train'

In [2]:
train_df1 = pd.read_json('data/train_word_transcripts.jsonl',lines=True)
train_df1['audio_path'] = 'data/'+train_df1['audio_path']
train_df1

,utterance_id,child_id,session_id,audio_path,audio_duration_sec,age_bucket,md5_hash,filesize_bytes,orthographic_text
0,U_00003c3ae1c35c6f,C_c74bfde2cca8d5da,S_7d821c3e4d3bc616,data/audio/U_00003c3ae1c35c6f.flac,1.920,8-11,9214be45ba2928dd57384f3c7ee54236,30672,hm
1,U_00003db24218ffe4,C_c74bfde2cca8d5da,S_e6103ab3a4538d71,data/audio/U_00003db24218ffe4.flac,12.737,8-11,fe761bb3d034530ef05163c7ad98ec3e,180942,yeah its pouring the water on the screen but t...
2,U_0001a0d0a3b4d816,C_4d0e1c16566d65a2,S_179057c3c3ccdecf,data/audio/U_0001a0d0a3b4d816.flac,11.556,8-11,b05073e65a98368fccbe777b5ab35e02,208352,it got water and sunlight but the plant did di...
3,U_00021d201a31d313,C_3b51c8b1d2c076d8,S_90720887e4430996,data/audio/U_00021d201a31d313.flac,1.125,8-11,9ed95318724ae6a2d1ce95d6aa743f6b,27099,there is wires
4,U_0003537f2bc1eb0b,C_b50216b3c70ca0a2,S_5b0bb48fadd7f802,data/audio/U_0003537f2bc1eb0b.flac,1.125,8-11,f3142751c6a52e2c24a85a4544fe8a0f,18476,good
...,...,...,...,...,...,...,...,...,...
95567,U_fffb5523ed9ea68f,C_c83f7ae3cffb9886,S_d655267f8a31b651,data/audio/U_fffb5523ed9ea68f.flac,0.336,8-11,c7853e8f492c21f7f35b29ea997976dc,14658,thank you
95568,U_fffd3181ab3dbb35,C_eb21c5635ec2a5fb,S_42e77ca587ff78c8,data/audio/U_fffd3181ab3dbb35.flac,6.217,8-11,4fe7f0926491d13fc5599d0b3be8a5d2,110716,the wax is going off and the fire's moving around
95569,U_fffd9ee77192956d,C_29528791cd81a931,S_75d13a9058485876,data/audio/U_fffd9ee77192956d.flac,22.759,8-11,617363baa65950b9adae023ccad29036,496115,well the reason they're dimmer is because um t...
95570,U_fffe57f6f30a4915,C_6354527999b15da1,S_8b1d247b058f5073,data/audio/U_fffe57f6f30a4915.flac,2.184,3-4,aaa0a3724331acdae526bc10c20361d7,178906,i think this go right here


In [3]:
train_df2 = pd.read_json('data/TalkBank_corpus/train_word_transcripts.jsonl',lines=True)
train_df2['audio_path'] = 'data/TalkBank_corpus/'+train_df2['audio_path']
train_df2

,utterance_id,child_id,session_id,audio_path,audio_duration_sec,age_bucket,md5_hash,filesize_bytes,orthographic_text
0,U_00007df27e1aad33,C_051ec7d3c1c79ea0,S_9fa42bab5070de99,/home/ubuntu/Desktop/audio/audio/U_00007df27e1...,0.656,12+,3c06f408d812a597f0bc40551e88587c,21843,er
1,U_0000891c5dd8a029,C_bb6d9ae9daeb252f,S_c0372465b6a6d449,/home/ubuntu/Desktop/audio/audio/U_0000891c5dd...,0.615,8-11,bed60778ecb0512df8b9ed8b87a2dbc4,40091,azoo
2,U_000115d21a4e9803,C_8cb9e2715147169e,S_e040bd862833a42e,/home/ubuntu/Desktop/audio/audio/U_000115d21a4...,1.095,8-11,026c0732f280756a145201e109614fab,53900,and then there's like a shop
3,U_00016e37f6f492d0,C_8bec179ceec547ea,S_16268a4120aaaf41,/home/ubuntu/Desktop/audio/audio/U_00016e37f6f...,1.457,8-11,1b37fdc261a6238afa5360db72d5b0ed,83925,um
4,U_00018bc22b7f1b7d,C_81d265d4c30a75c0,S_007b697d81ac2ec4,/home/ubuntu/Desktop/audio/audio/U_00018bc22b7...,0.938,3-4,0a844e8696b379c5f6700a9a1cd06360,38385,stars
...,...,...,...,...,...,...,...,...,...
255041,U_ffff47e74b3df4bd,C_eeef9cde2a489d3e,S_a4374ef38a06c9d3,/home/ubuntu/Desktop/audio/audio/U_ffff47e74b3...,4.651,3-4,993dadaabfc3c6a47cbd6e774251ca4c,199287,sock hop on pop top fox
255042,U_ffff5a85835f4381,C_c8dacd6170ef3dac,S_0513c82716cf293e,/home/ubuntu/Desktop/audio/audio/U_ffff5a85835...,0.750,8-11,63cbc529895536ede8643efd23ddd81f,64774,was it
255043,U_ffff9511f5573d86,C_19d203c2d174b50a,S_12ad2436005232a1,/home/ubuntu/Desktop/audio/audio/U_ffff9511f55...,6.264,5-7,f2dd7b7a0395cad6c6cdc8085cdb702c,274322,and you get to have a little break in there be...
255044,U_ffffcc5a06582c0f,C_528d220b627fbbf5,S_d96ff7b629fc2b49,/home/ubuntu/Desktop/audio/audio/U_ffffcc5a065...,0.951,3-4,ac2d88c0c6ef9a27d9ef338ae0c450fa,42610,orange


In [4]:
len(set(train_df1['child_id'])&set(train_df2['child_id']))

0

In [5]:
train_df = pd.concat([train_df1,train_df2]).reset_index(drop=True)
train_df

,utterance_id,child_id,session_id,audio_path,audio_duration_sec,age_bucket,md5_hash,filesize_bytes,orthographic_text
0,U_00003c3ae1c35c6f,C_c74bfde2cca8d5da,S_7d821c3e4d3bc616,data/audio/U_00003c3ae1c35c6f.flac,1.920,8-11,9214be45ba2928dd57384f3c7ee54236,30672,hm
1,U_00003db24218ffe4,C_c74bfde2cca8d5da,S_e6103ab3a4538d71,data/audio/U_00003db24218ffe4.flac,12.737,8-11,fe761bb3d034530ef05163c7ad98ec3e,180942,yeah its pouring the water on the screen but t...
2,U_0001a0d0a3b4d816,C_4d0e1c16566d65a2,S_179057c3c3ccdecf,data/audio/U_0001a0d0a3b4d816.flac,11.556,8-11,b05073e65a98368fccbe777b5ab35e02,208352,it got water and sunlight but the plant did di...
3,U_00021d201a31d313,C_3b51c8b1d2c076d8,S_90720887e4430996,data/audio/U_00021d201a31d313.flac,1.125,8-11,9ed95318724ae6a2d1ce95d6aa743f6b,27099,there is wires
4,U_0003537f2bc1eb0b,C_b50216b3c70ca0a2,S_5b0bb48fadd7f802,data/audio/U_0003537f2bc1eb0b.flac,1.125,8-11,f3142751c6a52e2c24a85a4544fe8a0f,18476,good
...,...,...,...,...,...,...,...,...,...
350613,U_ffff47e74b3df4bd,C_eeef9cde2a489d3e,S_a4374ef38a06c9d3,/home/ubuntu/Desktop/audio/audio/U_ffff47e74b3...,4.651,3-4,993dadaabfc3c6a47cbd6e774251ca4c,199287,sock hop on pop top fox
350614,U_ffff5a85835f4381,C_c8dacd6170ef3dac,S_0513c82716cf293e,/home/ubuntu/Desktop/audio/audio/U_ffff5a85835...,0.750,8-11,63cbc529895536ede8643efd23ddd81f,64774,was it
350615,U_ffff9511f5573d86,C_19d203c2d174b50a,S_12ad2436005232a1,/home/ubuntu/Desktop/audio/audio/U_ffff9511f55...,6.264,5-7,f2dd7b7a0395cad6c6cdc8085cdb702c,274322,and you get to have a little break in there be...
350616,U_ffffcc5a06582c0f,C_528d220b627fbbf5,S_d96ff7b629fc2b49,/home/ubuntu/Desktop/audio/audio/U_ffffcc5a065...,0.951,3-4,ac2d88c0c6ef9a27d9ef338ae0c450fa,42610,orange


In [6]:
# from joblib import Parallel, delayed
# import librosa
# def check_audio_file(file_path):
#     """
#     尝试加载一个音频文件。
#     如果成功，返回 None。
#     如果失败，返回文件路径。
#     """
#     try:
#         # 我们只需要检查是否能加载，不需要完整数据，可以设置duration来加速
#         # 如果文件很大，这个优化会非常有用
#         wav, _ = librosa.load(file_path, sr=16000, mono=True)
#         return None
#     except Exception as e:
#         # 打印错误信息是可选的，但有助于调试
#         print(f"无法加载文件 {file_path}: {e}")
#         return file_path


# # 2. 使用Joblib进行并行处理
# # 获取所有文件路径列表
# audio_paths = train_df['audio_path'].tolist()

# print("开始并行检查音频文件...")
# results = Parallel(n_jobs=30, backend='multiprocessing')(
#     delayed(check_audio_file)(path) for path in tqdm(audio_paths)
# )
# results = [i for i in results if i is not None]
# len(results)

In [7]:
child_ids = sorted(set(train_df['child_id']))

In [8]:
train_child_ids = child_ids[:len(child_ids)//10*9]

In [9]:
import numpy as np
train_df = train_df[train_df['audio_duration_sec']<20]
train_df = train_df[train_df['utterance_id']!='U_b8a4e8220e65219b'].reset_index(drop=True)
train_df['split']=np.where(train_df['child_id'].isin(train_child_ids),'train','split')
train_df

,utterance_id,child_id,session_id,audio_path,audio_duration_sec,age_bucket,md5_hash,filesize_bytes,orthographic_text,split
0,U_00003c3ae1c35c6f,C_c74bfde2cca8d5da,S_7d821c3e4d3bc616,data/audio/U_00003c3ae1c35c6f.flac,1.920,8-11,9214be45ba2928dd57384f3c7ee54236,30672,hm,train
1,U_00003db24218ffe4,C_c74bfde2cca8d5da,S_e6103ab3a4538d71,data/audio/U_00003db24218ffe4.flac,12.737,8-11,fe761bb3d034530ef05163c7ad98ec3e,180942,yeah its pouring the water on the screen but t...,train
2,U_0001a0d0a3b4d816,C_4d0e1c16566d65a2,S_179057c3c3ccdecf,data/audio/U_0001a0d0a3b4d816.flac,11.556,8-11,b05073e65a98368fccbe777b5ab35e02,208352,it got water and sunlight but the plant did di...,train
3,U_00021d201a31d313,C_3b51c8b1d2c076d8,S_90720887e4430996,data/audio/U_00021d201a31d313.flac,1.125,8-11,9ed95318724ae6a2d1ce95d6aa743f6b,27099,there is wires,train
4,U_0003537f2bc1eb0b,C_b50216b3c70ca0a2,S_5b0bb48fadd7f802,data/audio/U_0003537f2bc1eb0b.flac,1.125,8-11,f3142751c6a52e2c24a85a4544fe8a0f,18476,good,train
...,...,...,...,...,...,...,...,...,...,...
343790,U_ffff47e74b3df4bd,C_eeef9cde2a489d3e,S_a4374ef38a06c9d3,/home/ubuntu/Desktop/audio/audio/U_ffff47e74b3...,4.651,3-4,993dadaabfc3c6a47cbd6e774251ca4c,199287,sock hop on pop top fox,split
343791,U_ffff5a85835f4381,C_c8dacd6170ef3dac,S_0513c82716cf293e,/home/ubuntu/Desktop/audio/audio/U_ffff5a85835...,0.750,8-11,63cbc529895536ede8643efd23ddd81f,64774,was it,train
343792,U_ffff9511f5573d86,C_19d203c2d174b50a,S_12ad2436005232a1,/home/ubuntu/Desktop/audio/audio/U_ffff9511f55...,6.264,5-7,f2dd7b7a0395cad6c6cdc8085cdb702c,274322,and you get to have a little break in there be...,train
343793,U_ffffcc5a06582c0f,C_528d220b627fbbf5,S_d96ff7b629fc2b49,/home/ubuntu/Desktop/audio/audio/U_ffffcc5a065...,0.951,3-4,ac2d88c0c6ef9a27d9ef338ae0c450fa,42610,orange,train


In [10]:
valid_df=train_df[train_df['split']!=fold].reset_index(drop=True)
# train_df=train_df[train_df['split']==fold].reset_index(drop=True)

In [11]:
train_df.shape,valid_df.shape

((343795, 10), (38300, 10))

In [12]:
train_df

,utterance_id,child_id,session_id,audio_path,audio_duration_sec,age_bucket,md5_hash,filesize_bytes,orthographic_text,split
0,U_00003c3ae1c35c6f,C_c74bfde2cca8d5da,S_7d821c3e4d3bc616,data/audio/U_00003c3ae1c35c6f.flac,1.920,8-11,9214be45ba2928dd57384f3c7ee54236,30672,hm,train
1,U_00003db24218ffe4,C_c74bfde2cca8d5da,S_e6103ab3a4538d71,data/audio/U_00003db24218ffe4.flac,12.737,8-11,fe761bb3d034530ef05163c7ad98ec3e,180942,yeah its pouring the water on the screen but t...,train
2,U_0001a0d0a3b4d816,C_4d0e1c16566d65a2,S_179057c3c3ccdecf,data/audio/U_0001a0d0a3b4d816.flac,11.556,8-11,b05073e65a98368fccbe777b5ab35e02,208352,it got water and sunlight but the plant did di...,train
3,U_00021d201a31d313,C_3b51c8b1d2c076d8,S_90720887e4430996,data/audio/U_00021d201a31d313.flac,1.125,8-11,9ed95318724ae6a2d1ce95d6aa743f6b,27099,there is wires,train
4,U_0003537f2bc1eb0b,C_b50216b3c70ca0a2,S_5b0bb48fadd7f802,data/audio/U_0003537f2bc1eb0b.flac,1.125,8-11,f3142751c6a52e2c24a85a4544fe8a0f,18476,good,train
...,...,...,...,...,...,...,...,...,...,...
343790,U_ffff47e74b3df4bd,C_eeef9cde2a489d3e,S_a4374ef38a06c9d3,/home/ubuntu/Desktop/audio/audio/U_ffff47e74b3...,4.651,3-4,993dadaabfc3c6a47cbd6e774251ca4c,199287,sock hop on pop top fox,split
343791,U_ffff5a85835f4381,C_c8dacd6170ef3dac,S_0513c82716cf293e,/home/ubuntu/Desktop/audio/audio/U_ffff5a85835...,0.750,8-11,63cbc529895536ede8643efd23ddd81f,64774,was it,train
343792,U_ffff9511f5573d86,C_19d203c2d174b50a,S_12ad2436005232a1,/home/ubuntu/Desktop/audio/audio/U_ffff9511f55...,6.264,5-7,f2dd7b7a0395cad6c6cdc8085cdb702c,274322,and you get to have a little break in there be...,train
343793,U_ffffcc5a06582c0f,C_528d220b627fbbf5,S_d96ff7b629fc2b49,/home/ubuntu/Desktop/audio/audio/U_ffffcc5a065...,0.951,3-4,ac2d88c0c6ef9a27d9ef338ae0c450fa,42610,orange,train


In [13]:
with open(path+f'feature/train_{version}.jsonl', 'w',  encoding='utf-8') as fw:
    for idx,audio in enumerate(train_df['audio_path']):
        s = {"audio": audio, "text": "language English<asr_text>"+train_df['orthographic_text'][idx]}
        s = json.dumps(s, ensure_ascii=False)
        fw.write(s + '\n')

In [14]:
with open(path+f'feature/valid_{version}.jsonl', 'w',  encoding='utf-8') as fw:
    for idx,audio in enumerate(valid_df['audio_path']):
        s = {"audio": audio, "text": "language English<asr_text>"+valid_df['orthographic_text'][idx]}
        s = json.dumps(s, ensure_ascii=False)
        fw.write(s + '\n')

In [ ]:
from IPython import get_ipython
get_ipython().system(f'''torchrun --nproc_per_node=4 qwen3_asr_sft.py \
  --model_path data/models/X4090D-v013/checkpoint-85950/ \
  --train_file data/feature/train_{version}.jsonl \
  --eval_file data/feature/valid_{version}.jsonl \
  --output_dir data/models/{version} \
  --batch_size 3 \
  --grad_acc 1 \
  --lr 1e-6 \
  --epochs 5 \
  --log_steps 10 \
  --save_strategy steps \
  --save_steps 28650 \
  --save_total_limit 5 \
  --num_workers 8 \
  --pin_memory 1 \
  --persistent_workers 1 \
  --prefetch_factor 2''')

W0320 16:36:22.201000 1729648 site-packages/torch/distributed/run.py:803] 
W0320 16:36:22.201000 1729648 site-packages/torch/distributed/run.py:803] *****************************************
W0320 16:36:22.201000 1729648 site-packages/torch/distributed/run.py:803] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0320 16:36:22.201000 1729648 site-packages/torch/distributed/run.py:803] *****************************************
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The module name  (originally ) is not a valid Python identifier. Please rename the original module to avoid import issues.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The module name  (